In [ ]:
import numpy as np
import time

def f(X):
    x1, x2 = X
    return (x1**2 + 2*x2 - 10)**2 + (x1 + x2**2 - 5)**2

def grad_f(X):
    x1, x2 = X
    df_dx1 = 2*(x1**2 + 2*x2 - 10)*(2*x1) + 2*(x1 + x2**2 - 5)
    df_dx2 = 2*(x1**2 + 2*x2 - 10)*(2) + 2*(x1 + x2**2 - 5)*(2*x2)
    return np.array([df_dx1, df_dx2])

In [ ]:
# 1. DFP С АДАПТИВНЫМ ПОИСКОМ
'''
H_new = H + (s·sᵀ)/(s·y) - (H·y·yᵀ·H)/(yᵀ·H·y)
s = x_new - x_old
y = ∇f(x_new) - ∇f(x_old)
'''
def line_search_armijo(f, grad_f, x, d, alpha0=1.0, c=0.5, rho=0.5):
    """Armijo rule для выбора шага"""
    g = grad_f(x)
    fx = f(x)
    alpha = alpha0
    while f(x + alpha * d) > fx + c * alpha * np.dot(g, d):
        alpha *= rho
    return alpha

def dfp_method(f, grad_f, x0, tol=1e-6, max_iter=100):
    n = len(x0)
    x = x0.copy()
    H = np.eye(n)
    f_calls = 0
    g_calls = 0
    
    def f_wrapped(x):
        nonlocal f_calls
        f_calls += 1
        return f(x)
    
    def grad_wrapped(x):
        nonlocal g_calls
        g_calls += 1
        return grad_f(x)
    
    for i in range(max_iter):
        g = grad_wrapped(x)
        if np.linalg.norm(g) < tol:
            break
        
        d = -H @ g
        alpha = line_search_armijo(f_wrapped, grad_wrapped, x, d)
        x_next = x + alpha * d
        
        # Критерии остановки
        if np.linalg.norm(x_next - x) < tol:
            x = x_next
            break
        if abs(f_wrapped(x_next) - f_wrapped(x)) < tol:
            x = x_next
            break
        
        s = x_next - x
        y = grad_wrapped(x_next) - g
        
        # Обновление матрицы H
        sy = np.dot(s, y)
        Hy = H @ y
        yHy = np.dot(y, Hy)
        
        if sy > 1e-12 and yHy > 1e-12:
            H = H + np.outer(s, s) / sy - np.outer(Hy, Hy) / yHy
        
        x = x_next
    
    return x, i, f_calls, g_calls

In [ ]:
#2. МЕТОД ПАУЭЛЛА
def golden_section_search(f, x, d, tol=1e-5, max_step=10):
    alpha = 0
    step = 0.1
    f0 = f(x)
    
    a, b = 0, step
    fa, fb = f0, f(x + b * d)
    while fb < fa and b < max_step:
        a, b = b, b * 2
        fa, fb = fb, f(x + b * d)
    
    if b == step and fb > fa:
        a, b = -step, 0
        fa, fb = f(x + a * d), f0
        while fa < fb and a > -max_step:
            b, a = a, a * 2
            fb, fa = fa, f(x + a * d)
        a, b = a, b  
        if a > b:
            a, b = b, a
            fa, fb = fb, fa
    
    gr = (np.sqrt(5) - 1) / 2
    for _ in range(50):
        if abs(b - a) < tol:
            break
        c = b - gr * (b - a)
        d_alpha = a + gr * (b - a)
        fc = f(x + c * d)
        fd = f(x + d_alpha * d)
        if fc < fd:
            b = d_alpha
        else:
            a = c
    return (a + b) / 2

def powell_manual(f, x0, tol=1e-6, max_iter=100):
    n = len(x0)
    x = np.array(x0, dtype=float)
    directions = np.eye(n)
    f_calls = 0
    
    def f_wrapped(x):
        nonlocal f_calls
        f_calls += 1
        return f(x)
    
    for iteration in range(max_iter):
        x_prev = x.copy()
        
        for i, d in enumerate(directions):
            alpha_opt = golden_section_search(f_wrapped, x, d)
            x = x + alpha_opt * d
        
        new_direction = x - x_prev
        if np.linalg.norm(new_direction) < tol:
            break
        if abs(f_wrapped(x) - f_wrapped(x_prev)) < tol:
            break
        
        directions[1:] = directions[:-1].copy()
        directions[0] = new_direction
    
    return x, iteration, f_calls


In [ ]:
# 3. МЕТОД КРОСС-ЭНТРОПИИ
def cross_entropy_method(f, x0, iterations=50, sample_size=200, elite_frac=0.1, restarts=3):
    n = len(x0)
    best_solution = None
    best_value = float('inf')
    total_f_calls = 0
    
    for restart in range(restarts):
        mu = x0.copy() + np.random.randn(n) * 0.5
        sigma = np.ones(n) * 2.0
        f_calls = 0
        
        def f_wrapped(x):
            nonlocal f_calls
            f_calls += 1
            return f(x)
        
        for i in range(iterations):
            samples = np.random.multivariate_normal(mu, np.diag(sigma**2), sample_size)
            costs = np.array([f_wrapped(s) for s in samples])
            
            elite_size = max(1, int(sample_size * elite_frac))
            elite_idx = np.argsort(costs)[:elite_size]
            elite_samples = samples[elite_idx]
            
            mu_new = np.mean(elite_samples, axis=0)
            sigma_new = np.std(elite_samples, axis=0) + 1e-6
            
            mu = 0.7 * mu_new + 0.3 * mu
            sigma = 0.7 * sigma_new + 0.3 * sigma
            
            if np.max(sigma) < 1e-4:
                break
        
        final_value = f_wrapped(mu)
        if final_value < best_value:
            best_value = final_value
            best_solution = mu.copy()
        
        total_f_calls += f_calls
    
    return best_solution, iterations * restarts, total_f_calls

In [10]:
# ТЕСТИРОВАНИЕ
def identify_minimum(x, tol=1e-2):
    min_A = np.array([3.525428, -1.214320])
    min_B = np.array([-2.156324, 2.675131])
    local_min = np.array([2.6309, 1.5392])  # локальный минимум
    
    dist_to_A = np.linalg.norm(x - min_A)
    dist_to_B = np.linalg.norm(x - min_B)
    dist_to_local = np.linalg.norm(x - local_min)
    
    if dist_to_A < tol:
        return "ГЛОБАЛЬНЫЙ МИНИМУМ A (3.525, -1.214)"
    elif dist_to_B < tol:
        return "ГЛОБАЛЬНЫЙ МИНИМУМ B (-2.156, 2.675)"
    elif dist_to_local < tol:
        return "ЛОКАЛЬНЫЙ МИНИМУМ (2.631, 1.539)"
    else:
        return f"ПРОМЕЖУТОЧНАЯ ТОЧКА (погрешность {min(dist_to_A, dist_to_B, dist_to_local):.3f})"

print("СРАВНЕНИЕ МЕТОДОВ ОПТИМИЗАЦИИ")

# Статистика по методам для отчёта
stats = {
    'DFP': {'f_calls': [], 'g_calls': [], 'time': [], 'found_global': 0},
    'Powell_manual': {'f_calls': [], 'time': [], 'found_global': 0},
    'CEM': {'f_calls': [], 'time': [], 'found_global': 0}
}

for x_start in starts:
    print(f"Начальная точка: {x_start}")
    
    results = {}
    
    # DFP
    start_t = time.time()
    res, iters, f_calls, g_calls = dfp_method(f, grad_f, x_start)
    min_type = identify_minimum(res)
    is_global = "ГЛОБАЛЬНЫЙ" in min_type
    results['DFP'] = {
        'x': res, 
        'f': f(res),
        'iters': iters, 
        'f_calls': f_calls,
        'g_calls': g_calls,
        'time': time.time() - start_t,
        'min_type': min_type,
        'is_global': is_global
    }
    stats['DFP']['f_calls'].append(f_calls)
    stats['DFP']['g_calls'].append(g_calls)
    stats['DFP']['time'].append(time.time() - start_t)
    if is_global:
        stats['DFP']['found_global'] += 1
    
    # Powell (manual)
    start_t = time.time()
    res, iters, f_calls = powell_manual(f, x_start)
    min_type = identify_minimum(res)
    is_global = "ГЛОБАЛЬНЫЙ" in min_type
    results['Powell_manual'] = {
        'x': res,
        'f': f(res),
        'iters': iters,
        'f_calls': f_calls,
        'g_calls': 0,
        'time': time.time() - start_t,
        'min_type': min_type,
        'is_global': is_global
    }
    stats['Powell_manual']['f_calls'].append(f_calls)
    stats['Powell_manual']['time'].append(time.time() - start_t)
    if is_global:
        stats['Powell_manual']['found_global'] += 1
    
    # CEM
    start_t = time.time()
    res, iters, f_calls = cross_entropy_method(f, x_start)
    min_type = identify_minimum(res)
    is_global = "ГЛОБАЛЬНЫЙ" in min_type
    results['CEM'] = {
        'x': res,
        'f': f(res),
        'iters': iters,
        'f_calls': f_calls,
        'g_calls': 0,
        'time': time.time() - start_t,
        'min_type': min_type,
        'is_global': is_global
    }
    stats['CEM']['f_calls'].append(f_calls)
    stats['CEM']['time'].append(time.time() - start_t)
    if is_global:
        stats['CEM']['found_global'] += 1
    
    # Вывод результатов
    for method, data in results.items():
        print(f"\n📌 {method}:")
        print(f"   Решение: x1={data['x'][0]:.6f}, x2={data['x'][1]:.6f}")
        print(f"   f(x) = {data['f']:.2e}")
        print(f"   Тип: {data['min_type']}")
        print(f"   Итераций: {data['iters']}")
        print(f"   Вызовов f: {data['f_calls']}")
        if data['g_calls'] > 0:
            print(f"   Вызовов grad: {data['g_calls']}")
        print(f"   Время: {data['time']:.4f} с")
    
    # Вывод сходимости
    print(f"\n--- СХОДИМОСТЬ МЕТОДОВ ---")
    global_count = sum(1 for d in results.values() if d['is_global'])
    if global_count == len(results):
        print("🎉 ВСЕ МЕТОДЫ НАШЛИ ГЛОБАЛЬНЫЙ МИНИМУМ (A или B)!")
    elif global_count > 0:
        print(f"✅ {global_count} из {len(results)} методов нашли глобальный минимум")
        for method, data in results.items():
            if not data['is_global']:
                print(f"   ⚠️ {method} застрял в {data['min_type'].lower()}")
    else:
        print("⚠️ Ни один метод не нашёл глобальный минимум")
        print("   (все застряли в локальном минимуме)")


СРАВНЕНИЕ МЕТОДОВ ОПТИМИЗАЦИИ
Начальная точка: [0. 0.]

📌 DFP:
   Решение: x1=2.630820, x2=1.539287
   f(x) = 9.49e-08
   Тип: ЛОКАЛЬНЫЙ МИНИМУМ (2.631, 1.539)
   Итераций: 8
   Вызовов f: 46
   Вызовов grad: 26
   Время: 0.0010 с

📌 Powell_manual:
   Решение: x1=3.579886, x2=-1.268000
   f(x) = 1.13e-01
   Тип: ПРОМЕЖУТОЧНАЯ ТОЧКА (погрешность 0.076)
   Итераций: 22
   Вызовов f: 2467
   Время: 0.0057 с

📌 CEM:
   Решение: x1=2.630896, x2=1.539195
   f(x) = 2.99e-10
   Тип: ЛОКАЛЬНЫЙ МИНИМУМ (2.631, 1.539)
   Итераций: 150
   Вызовов f: 10803
   Время: 0.0151 с

--- СХОДИМОСТЬ МЕТОДОВ ---
⚠️ Ни один метод не нашёл глобальный минимум
   (все застряли в локальном минимуме)
Начальная точка: [5. 5.]

📌 DFP:
   Решение: x1=2.630878, x2=1.539211
   f(x) = 6.02e-09
   Тип: ЛОКАЛЬНЫЙ МИНИМУМ (2.631, 1.539)
   Итераций: 19
   Вызовов f: 94
   Вызовов grad: 59
   Время: 0.0004 с

📌 Powell_manual:
   Решение: x1=-2.159896, x2=2.684625
   f(x) = 3.42e-03
   Тип: ПРОМЕЖУТОЧНАЯ ТОЧКА (погрешность 0